# **Step 1: Setup Hugging Face API Token**

# **Step 1: Install Required Dependencies**

In [ ]:
!pip install unsloth # install unsloth
!pip install --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git # Also get the latest version Unsloth!

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.3/65.3 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 90.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.2/421.2 kB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 98.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 107.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 85.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.

# **Step 2: Import necessary Libraries**

In [ ]:
from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer
from unsloth import is_bfloat16_supported
from huggingface_hub import login
from transformers import TrainingArguments
from datasets import load_dataset
import wandb

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


# **Step 4: Check Hugging Face Token**

In [ ]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get("HF_TOKEN")
login(hf_token)

# **GPU Is Available?**

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

CUDA available: True
GPU device: Tesla T4


# **Step 5: Pretrained Deepseek R1**

In [ ]:
model_name = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
max_sequence_length = 1024
dtype = None
load_in_4bit = True

# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name=model_name,
#     # max_sequence_length=max_sequence_length,
#     dtype = dtype,
#     load_in_4bit=load_in_4bit,
#     token = hf_token
# )

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    load_in_4bit = load_in_4bit,
    max_seq_length = 1024,
    fast_inference = False,
    disable_log_stats = True,
    local_files_only = False,
)


==((====))==  Unsloth 2026.4.6: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.96G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/236 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth: Will load unsloth/deepseek-r1-distill-llama-8b-unsloth-bnb-4bit as a legacy tokenizer.


# **Step 6: Setup System Prompt**

In [ ]:

prompt_style = """
Below is a task description along with additional context provided in the input section. Your goal is to provide a well-reasoned response that effectively addresses the request.

Before crafting your answer, take a moment to carefully analyze the question. Develop a clear, step-by-step thought process to ensure your response is both logical and accurate.

### Task:
You are a medical expert specializing in clinical reasoning, diagnostics, and treatment planning. Answer the medical question below using your advanced knowledge.

### Query:
{}

### Answer:
<think>{}
"""

# **Step 7: Model Inferencing**

In [ ]:
# import warnings
# warnings.filterwarnings("ignore")

# from torch.nn import attention
# # Define a test question

# prompt = f"""<|user|> A 61-year-old woman with a long history of involuntary urine loss during activities like coughing or
#               sneezing but no leakage at night undergoes a gynecological exam and Q-tip test. Based on these findings,
#               what would cystometry most likely reveal about her residual volume and detrusor contractions?"""

# # FastLanguageModel.for_inference(model)

# # Tokenize the Input
# inputs = tokenizer(
#     prompt,
#     return_tensors="pt",
#     padding=True,
#     truncation=True
# ).to("cuda")

# # Generate a Response
# outputs = model.generate(
#     input_ids=inputs.input_ids,
#     attention_mask=inputs.attention_mask,
#     max_new_tokens=200,
#     pad_token_id=tokenizer.eos_token_id,
#     use_cache=True
# )


# # Decode the response tokens back to text
# response = tokenizer.batch_decode(outputs)

# # Print the response
# print(response)

In [ ]:
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name = model_name,
#     load_in_4bit = True,
#     max_seq_length = 2048,
#     fast_inference = False,
# )

model = model.eval()   # ✅ important

prompt = """<|user|>
A 61-year-old woman with a long history of involuntary urine loss during activities like coughing or sneezing but no leakage at night undergoes a gynecological exam and Q-tip test. Based on these findings, what would cystometry most likely reveal about her residual volume and detrusor contractions?


<|assistant|>
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    padding=True,
    truncation=True
).to("cuda")

outputs = model.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=200,
    pad_token_id=tokenizer.eos_token_id,
    do_sample=True,
    temperature=0.7,
    use_cache=False   # 🔥 KEY FIX
)

response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True,
    clean_up_tokenization_spaces=True   # ✅ important
)

response = response.replace("Ġ", " ")

print(response)

Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.1

<|user|>A61-year-oldwomanwithalonghistoryofinvoluntaryurinelossduringactivitieslikecoughingorsneezingbutnoleakageatnightundergoesagynecologicalexamandQ-tiptest.Basedonthesefindings,whatwouldcystometrymostlikelyrevealaboutherresidualvolumeanddetrusorcontractions?<|assistant|>**Answer**Ċ**Residual Volume and Detrusor Contractions in a 61-Year-Old Woman with History of Involuntary Urine Loss**ĊA 61-year-old woman presents with a history of involuntary urine loss during activities such as coughing or sneezing but without leakage at night. She undergoes a gynecologic exam and Q-tip test. The question is asking what cystometry would most likely reveal about her residual volume and detrusor contractions.ĊĊFirst, let's break down the terms:Ċ- **Residual Volume (RV):** This is the volume of urine remaining in the bladder after a complete emptying effort.Ċ- **Detrusor Contractions:** These are involuntary contractions of the detrusor muscle (the muscle that surrounds the bladder), which can caus

In [ ]:
print(response.split('<|assistant|>')[1].lstrip().replace('ĊĊ', '\n\n'))

**Answer**Ċ**Residual Volume and Detrusor Contractions in a 61-Year-Old Woman with History of Involuntary Urine Loss**ĊA 61-year-old woman presents with a history of involuntary urine loss during activities such as coughing or sneezing but without leakage at night. She undergoes a gynecologic exam and Q-tip test. The question is asking what cystometry would most likely reveal about her residual volume and detrusor contractions.

First, let's break down the terms:Ċ- **Residual Volume (RV):** This is the volume of urine remaining in the bladder after a complete emptying effort.Ċ- **Detrusor Contractions:** These are involuntary contractions of the detrusor muscle (the muscle that surrounds the bladder), which can cause an increase in intra-bladder pressure, leading to urination.

Now, considering her history:Ċ- She experiences urine loss during activities like coughing or sneezing. This suggests that


In [ ]:
import re

# The `prompt` variable is available from the previous execution, containing the user's input for generation.
# Extract the actual user query part from the `prompt` variable, which is known to be well-formatted.
user_query_text = prompt.split('<|user|>')[1].split('<|assistant|>')[0].strip()
# Clean up any potential extra newlines or multiple spaces that might have been part of the prompt string's formatting
user_query_text = re.sub(r'\n\s*', ' ', user_query_text).strip()
user_query_text = re.sub(r' +', ' ', user_query_text).strip()


# Decode the full raw output from the model
# We set clean_up_tokenization_spaces=False here, as we will handle spaces manually more precisely
raw_decoded_output = tokenizer.decode(outputs[0], skip_special_tokens=False, clean_up_tokenization_spaces=False)

# Remove the beginning of sentence token which often appears as <\uff5cbegin of sentence\uff5c>
raw_decoded_output = raw_decoded_output.replace('<｜begin of sentence｜>', '').strip()

# Split the raw decoded output to isolate the assistant's generated part
parts_of_decoded_output = raw_decoded_output.split('<|assistant|>', 1) # Split only once

cleaned_assistant_response = ""
if len(parts_of_decoded_output) > 1:
    assistant_raw_content = parts_of_decoded_output[1]

    # Clean up the assistant's content:
    # 1. Replace 'Ġ' tokens with actual spaces
    cleaned_assistant_response = assistant_raw_content.replace('Ġ', ' ')

    # 2. Apply regex to insert spaces where words or numbers might have been merged in the assistant's output
    cleaned_assistant_response = re.sub(r'([a-z])([A-Z])', r'\1 \2', cleaned_assistant_response)
    cleaned_assistant_response = re.sub(r'([a-zA-Z])([0-9])', r'\1 \2', cleaned_assistant_response)
    cleaned_assistant_response = re.sub(r'([0-9])([a-zA-Z])', r'\1 \2', cleaned_assistant_response)

    # 3. Replace 'ĊĊ' with actual newlines
    cleaned_assistant_response = cleaned_assistant_response.replace('ĊĊ', '\n\n').strip()

    # 4. Remove any extra spaces introduced by previous replacements
    cleaned_assistant_response = re.sub(r' +', ' ', cleaned_assistant_response).strip()


# Reconstruct the full output using the clean user query and the cleaned assistant response
final_full_output = f"<|user|> {user_query_text}\n<|assistant|>\n{cleaned_assistant_response}"

print("--- Cleaned Full Model Output (Input + Response) ---")
print(final_full_output)

print("\n--- Extracted Assistant's Response ---")
print(cleaned_assistant_response)

--- Cleaned Full Model Output (Input + Response) ---
<|user|> A 61-year-old woman with a long history of involuntary urine loss during activities like coughing or sneezing but no leakage at night undergoes a gynecological exam and Q-tip test. Based on these findings, what would cystometry most likely reveal about her residual volume and detrusor contractions?
<|assistant|>
**Answer**Ċ**Residual Volume and Detrusor Contractions in a 61-Year-Old Woman with History of Involuntary Urine Loss**ĊA 61-year-old woman presents with a history of involuntary urine loss during activities such as coughing or sneezing but without leakage at night. She undergoes a gynecologic exam and Q-tip test. The question is asking what cystometry would most likely reveal about her residual volume and detrusor contractions.

First, let's break down the terms:Ċ- **Residual Volume (RV):** This is the volume of urine remaining in the bladder after a complete emptying effort.Ċ- **Detrusor Contractions:** These are in

# **Step 8: Fine-Tuning**

In [ ]:
# Load Dataset
medical_dataset = load_dataset("FreedomIntelligence/medical-o1-reasoning-SFT", "en", split = "train[:500]")


README.md: 0.00B [00:00, ?B/s]

medical_o1_sft.json:   0%|          | 0.00/58.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/19704 [00:00<?, ? examples/s]

In [ ]:
medical_dataset[1]

{'Question': 'A 33-year-old woman is brought to the emergency department 15 minutes after being stabbed in the chest with a screwdriver. Given her vital signs of pulse 110/min, respirations 22/min, and blood pressure 90/65 mm Hg, along with the presence of a 5-cm deep stab wound at the upper border of the 8th rib in the left midaxillary line, which anatomical structure in her chest is most likely to be injured?',
 'Complex_CoT': "Okay, let's figure out what's going on here. A woman comes in with a stab wound from a screwdriver. It's in her chest, upper border of the 8th rib, left side, kind of around the midaxillary line. First thought, that's pretty close to where the lung sits, right?\n\nLet's talk about location first. This spot is along the left side of her body. Above the 8th rib, like that, is where a lot of important stuff lives, like the bottom part of the left lung, possibly the diaphragm too, especially considering how deep the screwdriver went.\n\nThe wound is 5 cm deep. Tha

In [ ]:
EOS_TOKEN = tokenizer.eos_token
EOS_TOKEN

'<｜end▁of▁sentence｜>'

In [ ]:
#Update the prompt style for training
train_prompt_style = """
Below is a task description along with additional context provided in the input section. Your goal is to provide a well-reasoned response that effectively addresses the request.

Before crafting your answer, take a moment to carefully analyze the question. Develop a clear, step-by-step thought process to ensure your response is both logical and accurate.

### Instructions:
You are a medical expert specializing in clinical reasoning, diagnostics, and treatment planning. Answer the medical question below using your advanced knowledge.

### Question:
{}

### Response:
<think>
{}
</think>
{}
"""

In [ ]:
train_prompt_style = """<|user|>
{}

<|assistant|>
<think>
{}
</think>

{}
"""

In [ ]:
def preprocess_input_data(examples):
    inputs = examples["Question"]
    cots = examples["Complex_CoT"]
    outputs = examples["Response"]

    texts = []

    for inp, cot, out in zip(inputs, cots, outputs):
        text = train_prompt_style.format(inp, cot, out) + EOS_TOKEN
        texts.append(text)

    return {
        "text": texts
    }

In [ ]:
finetune_dataset = medical_dataset.map(
    preprocess_input_data,
    batched=True
)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [ ]:
print(finetune_dataset.column_names)

['Question', 'Complex_CoT', 'Response', 'text']


In [ ]:
finetune_dataset["text"][0]

"<|user|>\nGiven the symptoms of sudden weakness in the left arm and leg, recent long-distance travel, and the presence of swollen and tender right lower leg, what specific cardiac abnormality is most likely to be found upon further evaluation that could explain these findings?\n\n<|assistant|>\n<think>\nOkay, let's see what's going on here. We've got sudden weakness in the person's left arm and leg - and that screams something neuro-related, maybe a stroke?\n\nBut wait, there's more. The right lower leg is swollen and tender, which is like waving a big flag for deep vein thrombosis, especially after a long flight or sitting around a lot.\n\nSo, now I'm thinking, how could a clot in the leg end up causing issues like weakness or stroke symptoms?\n\nOh, right! There's this thing called a paradoxical embolism. It can happen if there's some kind of short circuit in the heart - like a hole that shouldn't be there.\n\nLet's put this together: if a blood clot from the leg somehow travels to 

# **Step 9: Applying Lora Fine-Tuning to the Model**

In [ ]:
model_lora = FastLanguageModel.get_peft_model(
    model = model,
    r = 8,
    target_modules = [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    lora_alpha = 16, #lora_alpha indicated how much the adapters influence the model output.
    lora_dropout = 0, #means no dropout will be applied to model layers.
    bias = "none",
    use_gradient_checkpointing= "unsloth",
    random_state = 3047,
    use_rslora= False,
    loftq_config=None
)

Unsloth 2026.4.6 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [ ]:
trainer = SFTTrainer(
    model = model_lora,
    tokenizer = tokenizer,
    train_dataset = finetune_dataset,
    dataset_text_field = "text",
    max_seq_length = 1024,
    dataset_num_proc = 1,

    # Define Training Arguments
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        num_train_epochs = 1,
        warmup_steps = 5,
        max_steps = 200,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        report_to="wandb",
        logging_steps = 5,
        logging_strategy = "steps",
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir = "outputs",
    ),

)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/500 [00:00<?, ? examples/s]

**Clearing Memory**

In [ ]:
import torch
torch.cuda.empty_cache()

**Setting WanDB**

In [ ]:
!pip install weave

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.9/45.9 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 92.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.9/52.9 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 131.6 MB/s eta 0:00:00


In [ ]:
# Setup WANDB
# !pip install weave
import weave
from google.colab import userdata
wnb_token = userdata.get("WANDB_API_TOKEN")
# Login to WnB
wandb.login(key=wnb_token) # import wandb
run = wandb.init(
    project='Fine-tune-DeepSeek-R1-on-Medical-CoT-Dataset',
    job_type="training",
    anonymous="allow"
)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 221980030 (221980030-gift-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.


wandb: Initializing weave.
weave: wandb version 0.26.0 is available!  To upgrade, please run:
weave:  $ pip install wandb --upgrade
weave: Logged in as Weights & Biases user: 221980030.
weave: View Weave data at https://wandb.ai/221980030-gift-university/Fine-tune-DeepSeek-R1-on-Medical-CoT-Dataset/weave


In [ ]:
torch.compile = False

In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 4 | Total steps = 200
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 20,971,520 of 8,051,232,768 (0.26% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
5,2.853633
10,2.456138
15,2.177729
20,2.046726
25,2.040663
30,1.996244
35,1.996269
40,1.891708
45,1.968800
50,1.913304


# **Step 10: Testing after Fine-Tuning**

In [ ]:
def format_prompt(question):
    return f"""<|user|>
{question}

<|assistant|>
"""

In [ ]:
def generate_response(question):
    prompt = format_prompt(question)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        padding=True
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_new_tokens=300,
            do_sample=True,
            temperature=0.6,     # 🔥 lower = more accurate
            top_p=0.9,
            repetition_penalty=1.1,  # 🔥 reduces nonsense
            pad_token_id=tokenizer.eos_token_id,
            use_cache=False
        )

    response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True,
    clean_up_tokenization_spaces=True
    )

    # Extra cleanup
    response = response.replace("Ġ", " ")

    return response

In [ ]:
question = """A 61-year-old woman with urine leakage during coughing and sneezing but not at night. What would cystometry show?"""

response = generate_response(question)

print(response)

Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


<|user|>A61-year-oldwomanwithurineleakageduringcoughingandsneezingbutnotatnight.Whatwouldcystometryshow?<|assistant|><think>Alright,let'sthinkaboutthis.So,thisisawomanwho'shavingurineleakageswhenit'ssomethinglikeacoughorsneeze,butitdoesn'thappenatnight.Hmm,thatpatternisinteresting.Itmakessenseifwe'relookingatsomethingrelatedtoabdominalpressure,whichmightbethecasehere.Let'stalkaboutwhatcouldcausethiskindofincontinence.WhenIthinkofconditionswherethishappens,itoftenboilsdowntoanissuewiththebladderorurethra.Oh,right!Stressurinaryincontinencetendstoshowupinsituationslikethese.It'susuallyduetoaproblemwiththebladdermaybeoveractivityofthesmoothmuscles,ormaybeit'sjustabouttheflatnessoftheabdomenincreasingpressuresonthebladder.Now,cystometryisaprettykeytesthere.Theideaistotesthowmuchurineleaksoutduringacontrolledrhythmofbladdercontractions.Thiswillhelpdistinguishbetweenstressandnon-stresstypeofincontinence.Ifthere'sastressincontinencecomponent,thenweshouldseeareliableamountofurinedischargingeven

### Integrating Gradio for a User-Friendly Chatbot Interface

Gradio allows you to quickly create customizable web interfaces for your models with minimal code. It's great for sharing your models or building internal tools without diving deep into front-end development. Here's how you can integrate it with your fine-tuned chatbot model.

In [ ]:
# Install Gradio
!pip install -q gradio

In [ ]:
import gradio as gr

def chatbot_interface(message, history):
    # The generate_response function already formats the prompt correctly
    bot_response = generate_response(message)

    # The history parameter in Gradio's ChatInterface expects a list of lists,
    # where each inner list contains [user_message, bot_message].
    # For this simple integration, we'll just return the current response,
    # and Gradio's ChatInterface will handle the history accumulation automatically.
    return bot_response

# Create the Gradio ChatInterface
demo = gr.ChatInterface(
    fn=chatbot_interface,
    title="Fine-tuned DeepSeek-R1 Medical Chatbot",
    description="Ask the medical chatbot questions based on clinical reasoning.",
    examples=[
        "A 61-year-old woman with urine leakage during coughing and sneezing but not at night. What would cystometry show?",
        "A 33-year-old woman is brought to the emergency department 15 minutes after being stabbed in the chest with a screwdriver. Given her vital signs of pulse 110/min, respirations 22/min, and blood pressure 90/65 mm Hg, along with the presence of a 5-cm deep stab wound at the upper border of the 8th rib in the left midaxillary line, which anatomical structure in her chest is most likely to be injured?"
    ],
    theme="soft"
)

# Launch the Gradio interface
# Set share=True to get a public link (useful for sharing outside Colab)
# Set debug=True for more verbose output during development
demo.launch(share=True, debug=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://4742edafc0e9f86084.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=